In [ ]:

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras import Model

print("TensorFlow:", tf.__version__)

# ========================== CONFIG ==========================
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
SSL_EPOCHS = 30
TEMPERATURE = 0.5
AUTOTUNE = tf.data.AUTOTUNE
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"

# ========================== LOAD DATA ==========================
image_paths = []
labels = []

class_names = sorted(os.listdir(COMBINED_PATH))
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in os.listdir(folder):
        image_paths.append(os.path.join(folder, img))
        labels.append(class_to_idx[cls])

num_classes = len(class_names)

print("Classes:", num_classes)
print("Images :", len(image_paths))
print(class_names)

# ========================== TRAIN/VAL SPLIT ==========================
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=SEED, stratify=labels
)

print(len(train_paths))
print(len(val_paths))

# ========================== DATA LOADING FUNCTION ==========================
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    return img, label

# ========================== CREATE DATASET ==========================
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(load_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.shuffle(2000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(AUTOTUNE)

# ========================== AUGMENTATION ==========================
augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1)
])

# ========================== SIMCLR MODEL ==========================
base_encoder = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_encoder.trainable = True

inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_encoder(x, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
projection = layers.Dense(128)(x)

simclr_model = Model(inputs, projection)
simclr_model.summary()

# ========================== NT-XENT LOSS ==========================
def nt_xent_loss(zis, zjs, temperature=0.5):
    zis = tf.math.l2_normalize(zis, axis=1)
    zjs = tf.math.l2_normalize(zjs, axis=1)

    representations = tf.concat([zis, zjs], axis=0)
    similarity_matrix = tf.matmul(representations, representations, transpose_b=True)

    batch_size = tf.shape(zis)[0]
    mask = tf.eye(2 * batch_size)
    similarity_matrix = similarity_matrix - mask * 1e9

    positives = tf.concat([
        tf.range(batch_size, 2 * batch_size),
        tf.range(0, batch_size)
    ], axis=0)

    logits = similarity_matrix / temperature
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        positives, logits, from_logits=True
    )
    return tf.reduce_mean(loss)

optimizer = tf.keras.optimizers.Adam(1e-4)

# ========================== TRAIN STEP ==========================
@tf.function
def train_step(images):
    aug1 = augmenter(images)
    aug2 = augmenter(images)

    with tf.GradientTape() as tape:
        z1 = simclr_model(aug1, training=True)
        z2 = simclr_model(aug2, training=True)
        loss = nt_xent_loss(z1, z2, TEMPERATURE)

    grads = tape.gradient(loss, simclr_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, simclr_model.trainable_variables))
    return loss

# ========================== TRAINING LOOP ==========================
ssl_losses = []

for epoch in range(SSL_EPOCHS):
    batch_losses = []
    for images, _ in train_ds:
        loss = train_step(images)
        batch_losses.append(loss.numpy())

    epoch_loss = np.mean(batch_losses)
    ssl_losses.append(epoch_loss)

    print(f"Epoch {epoch+1}/{SSL_EPOCHS}  Loss={epoch_loss:.4f}")

# ========================== PLOT & SAVE ==========================
plt.figure(figsize=(8, 5))
plt.plot(ssl_losses)
plt.title("SimCLR NT-Xent Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.savefig("simclr_loss.png", dpi=600)
plt.show()

base_encoder.save("ssl_encoder.keras")
print("Encoder Saved")

In [ ]:
import os
import random
import hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from tensorflow.keras import layers, Model
from PIL import Image
import imagehash

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
SSL_EPOCHS = 30
CLASSIFIER_EPOCHS = 30
TEMPERATURE = 0.5
AUTOTUNE = tf.data.AUTOTUNE
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

image_paths = []
labels = []
class_names = sorted([
    d for d in os.listdir(COMBINED_PATH)
    if os.path.isdir(os.path.join(COMBINED_PATH, d))
])
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in sorted(os.listdir(folder)):
        path = os.path.join(folder, img)
        if os.path.isfile(path):
            image_paths.append(path)
            labels.append(class_to_idx[cls])

image_paths = np.array(image_paths)
labels = np.array(labels)
num_classes = len(class_names)

print("Classes:", num_classes)
print("Total images:", len(image_paths))
print("Class names:", class_names)
print("\nClass distribution:")
print(pd.Series(labels).map({v:k for k,v in class_to_idx.items()}).value_counts().sort_index())

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Train      :", len(train_paths))
print("Validation :", len(val_paths))
print("Test       :", len(test_paths))
print("Total      :", len(train_paths) + len(val_paths) + len(test_paths))
print("\nExpected proportions:")
print("Train      :", len(train_paths) / len(image_paths))
print("Validation :", len(val_paths) / len(image_paths))
print("Test       :", len(test_paths) / len(image_paths))

train_set = set(train_paths)
val_set = set(val_paths)
test_set = set(test_paths)

print("\nPath overlap:")
print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test      :", len(train_set & test_set))
print("Validation ∩ Test :", len(val_set & test_set))

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(
        img, channels=3, expand_animations=False
    )
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
).map(load_image, num_parallel_calls=AUTOTUNE)

train_ds = (
    train_ds
    .shuffle(len(train_paths), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
).map(load_image, num_parallel_calls=AUTOTUNE)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
).map(load_image, num_parallel_calls=AUTOTUNE)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets created successfully.")

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def perceptual_hash(path):
    with Image.open(path) as img:
        return imagehash.phash(img)

def duplicate_audit(paths_a, paths_b):
    md5_a = {}
    for p in paths_a:
        md5_a.setdefault(md5_file(p), []).append(p)

    md5_b = {}
    for p in paths_b:
        md5_b.setdefault(md5_file(p), []).append(p)

    common_md5 = set(md5_a) & set(md5_b)
    byte_identical_pairs = [
        (pa, pb)
        for h in common_md5
        for pa in md5_a[h]
        for pb in md5_b[h]
    ]

    hashes_a = [(p, perceptual_hash(p)) for p in paths_a]
    hashes_b = [(p, perceptual_hash(p)) for p in paths_b]

    near_pairs = []
    affected_b = set()
    for pa, ha in hashes_a:
        for pb, hb in hashes_b:
            dist = ha - hb
            if dist <= 5:
                near_pairs.append((pa, pb, dist))
                affected_b.add(pb)

    return byte_identical_pairs, near_pairs, affected_b

pairs = [
    ("Train-Validation", train_paths, val_paths),
    ("Train-Test", train_paths, test_paths),
    ("Validation-Test", val_paths, test_paths)
]

audit_rows = []
for name, a, b in pairs:
    exact_path_overlap = len(set(a) & set(b))
    byte_pairs, near_pairs, affected_b = duplicate_audit(a, b)
    audit_rows.append({
        "Comparison": name,
        "Exact path overlap": exact_path_overlap,
        "Byte-identical MD5 pairs": len(byte_pairs),
        "Near-duplicate pairs (pHash <= 5)": len(near_pairs),
        "Affected images in second partition": len(affected_b),
        "Affected percentage": (
            100 * len(affected_b) / len(b) if len(b) else 0
        )
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
audit_df.to_csv("three_way_split_integrity_audit.csv", index=False)
print("\nAudit saved: three_way_split_integrity_audit.csv")

augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1)
])

base_encoder = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_encoder.trainable = True

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_encoder(x, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
projection = layers.Dense(128)(x)

simclr_model = Model(inputs, projection)
simclr_model.summary()

def nt_xent_loss(zis, zjs, temperature=0.5):
    zis = tf.math.l2_normalize(zis, axis=1)
    zjs = tf.math.l2_normalize(zjs, axis=1)
    representations = tf.concat([zis, zjs], axis=0)
    similarity_matrix = tf.matmul(
        representations,
        representations,
        transpose_b=True
    )
    batch_size = tf.shape(zis)[0]
    mask = tf.eye(2 * batch_size)
    similarity_matrix = similarity_matrix - mask * 1e9
    positives = tf.concat([
        tf.range(batch_size, 2 * batch_size),
        tf.range(0, batch_size)
    ], axis=0)
    logits = similarity_matrix / temperature
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        positives,
        logits,
        from_logits=True
    )
    return tf.reduce_mean(loss)

optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images):
    aug1 = augmenter(images)
    aug2 = augmenter(images)
    with tf.GradientTape() as tape:
        z1 = simclr_model(aug1, training=True)
        z2 = simclr_model(aug2, training=True)
        loss = nt_xent_loss(z1, z2, TEMPERATURE)
    grads = tape.gradient(
        loss,
        simclr_model.trainable_variables
    )
    optimizer.apply_gradients(
        zip(grads, simclr_model.trainable_variables)
    )
    return loss

ssl_losses = []
for epoch in range(SSL_EPOCHS):
    batch_losses = []
    for images, _ in train_ds:
        loss = train_step(images)
        batch_losses.append(loss.numpy())
    epoch_loss = np.mean(batch_losses)
    ssl_losses.append(epoch_loss)
    print(
        f"Epoch {epoch + 1}/{SSL_EPOCHS} "
        f"Loss={epoch_loss:.4f}"
    )

plt.figure(figsize=(8, 5))
plt.plot(ssl_losses)
plt.title("SimCLR NT-Xent Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.savefig("simclr_loss.png", dpi=600)
plt.show()

base_encoder.save("ssl_encoder.keras")
print("SSL encoder saved.")

feature_extractor = tf.keras.models.load_model(
    "ssl_encoder.keras"
)

class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)

    shared_dense_1 = layers.Dense(
        channel // ratio,
        activation="relu"
    )
    shared_dense_2 = layers.Dense(channel)

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Add()([avg_out, max_out])
    channel_attention = layers.Activation(
        "sigmoid"
    )(channel_attention)

    channel_refined = layers.Multiply()([
        input_feature,
        channel_attention
    ])

    avg_pool_spatial = ChannelMean()(channel_refined)
    max_pool_spatial = ChannelMax()(channel_refined)
    concat = layers.Concatenate(axis=-1)([
        avg_pool_spatial,
        max_pool_spatial
    ])

    spatial_attention = layers.Conv2D(
        filters=1,
        kernel_size=7,
        padding="same",
        activation="sigmoid"
    )(concat)

    refined_feature = layers.Multiply()([
        channel_refined,
        spatial_attention
    ])

    return refined_feature

NUM_CLASSES = len(class_names)

inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)
x = tf.keras.applications.mobilenet_v2.preprocess_input(
    inputs
)
x = feature_extractor(x, training=False)
x = cbam_block(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(
    512,
    activation="relu",
    name="feature_dense"
)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode="min",
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    mode="min",
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CLASSIFIER_EPOCHS,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ]
)

model.load_weights("best_model.weights.h5")
print("Best validation-loss checkpoint loaded.")

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_curve.png", dpi=600)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=600)
plt.show()

test_loss, test_acc = model.evaluate(
    test_ds,
    verbose=1
)

print("\n==============================")
print("FINAL UNTOUCHED TEST RESULT")
print("==============================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

y_true = []
y_pred = []
y_prob = []

for images, labels_batch in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("Test prediction completed.")
print("Test samples:", len(y_true))

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)
print(report)

with open(
    "test_classification_report.txt",
    "w"
) as f:
    f.write(report)

test_acc = accuracy_score(y_true, y_pred)
test_precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)
test_recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)
test_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("==============================")
print("FINAL TEST METRICS")
print("==============================")
print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1 Score  : {test_f1:.4f}")

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        test_acc,
        test_precision,
        test_recall,
        test_f1
    ]
})

metrics_df.to_csv(
    "final_test_metrics.csv",
    index=False
)

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(10, 8))
plt.imshow(cm)
plt.colorbar()
plt.xticks(
    np.arange(len(class_names)),
    class_names,
    rotation=90
)
plt.yticks(
    np.arange(len(class_names)),
    class_names
)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Confusion Matrix — Untouched Test Set")
plt.tight_layout()
plt.savefig(
    "test_confusion_matrix.png",
    dpi=600
)
plt.show()

n_classes = len(class_names)
y_true_bin = label_binarize(
    y_true,
    classes=np.arange(n_classes)
)

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(
        y_true_bin[:, i],
        y_prob[:, i]
    )
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr,
        tpr,
        label=f"{class_names[i]} (AUC={roc_auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-Class ROC Curve — Untouched Test Set")
plt.legend(
    bbox_to_anchor=(1.05, 1)
)
plt.tight_layout()
plt.savefig(
    "test_roc_curve.png",
    dpi=300
)
plt.show()

feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("feature_dense").output
)

features = feature_model.predict(
    test_ds,
    verbose=1
)
print("Feature Shape:", features.shape)

perplexity = min(30, max(5, (len(features) - 1) // 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    random_state=SEED,
    init="pca"
)
features_2d = tsne.fit_transform(features)

plt.figure(figsize=(15, 10))
for i, class_name in enumerate(class_names):
    idx = np.where(y_true == i)[0]
    plt.scatter(
        features_2d[idx, 0],
        features_2d[idx, 1],
        label=class_name,
        alpha=0.7
    )

plt.title(
    "t-SNE Visualization of Test Feature Embeddings"
)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=8
)
plt.tight_layout()
plt.savefig(
    "test_tsne_class_names.png",
    dpi=600
)
plt.show()

model.save(
    "AgriSSL_CBAMNet_final_test_evaluated.keras"
)
print("Final model saved.")
print("Test set was used only for final evaluation.")

print("================================================")
print("FINAL EXPERIMENT SUMMARY")
print("================================================")
print(f"Total images       : {len(image_paths)}")
print(f"Training images    : {len(train_paths)}")
print(f"Validation images  : {len(val_paths)}")
print(f"Test images        : {len(test_paths)}")
print("\nModel-selection set : Validation")
print("Final evaluation set: Untouched Test")
print("\nDuplicate integrity:")
display(audit_df)
print("\nFinal test metrics:")
display(metrics_df)

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

os.makedirs("output", exist_ok=True)

# ========================== IDENTIFY MISCLASSIFIED SAMPLES ==========================
misclassified_idx = np.where(y_true != y_pred)[0]

print(f"Total Validation Samples : {len(y_true)}")
print(f"Correctly Classified     : {len(y_true) - len(misclassified_idx)}")
print(f"Misclassified            : {len(misclassified_idx)}")
print(f"Misclassification Rate   : {len(misclassified_idx)/len(y_true)*100:.2f}%")

# ========================== VISUALIZE MISCLASSIFIED IMAGES ==========================
def visualize_misclassified(n=12):
    n = min(n, len(misclassified_idx))
    sample_idx = misclassified_idx[:n]

    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, idx in enumerate(sample_idx):
        img = tf.io.read_file(val_paths[idx])
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.cast(img, tf.uint8)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(
            f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]} "
            f"({np.max(y_prob[idx])*100:.1f}%)",
            fontsize=9, color="red"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.savefig("output/misclassified_examples.png", dpi=600, bbox_inches="tight")
    plt.show()

visualize_misclassified(n=12)

In [ ]:

!pip install imagehash --quiet

import os
import hashlib
import imagehash
from PIL import Image
import pandas as pd
import numpy as np
from collections import defaultdict

OUTPUT_DIR = "/kaggle/working"         
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- 1. EXACT PATH / FILENAME OVERLAP ----------
train_set = set(train_paths)
val_set = set(val_paths)

exact_path_overlap = train_set & val_set
print(f"Exact path overlap (same file in both splits): {len(exact_path_overlap)}")

train_fnames = set(os.path.basename(p) for p in train_paths)
val_fnames = set(os.path.basename(p) for p in val_paths)
fname_overlap = train_fnames & val_fnames
print(f"Filename overlap (possible copy-with-different-folder): {len(fname_overlap)}")

# ---------- 2. EXACT BYTE-LEVEL DUPLICATE CHECK (MD5) ----------
def md5_of_file(path, chunk_size=8192):
    h = hashlib.md5()
    try:
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

print("\nComputing MD5 hashes for exact byte-duplicate check...")
train_md5 = {p: md5_of_file(p) for p in train_paths}
val_md5 = {p: md5_of_file(p) for p in val_paths}

train_md5_to_path = defaultdict(list)
for p, h in train_md5.items():
    if h:
        train_md5_to_path[h].append(p)

exact_byte_dupes = []
for vp, vh in val_md5.items():
    if vh in train_md5_to_path:
        for tp in train_md5_to_path[vh]:
            exact_byte_dupes.append((tp, vp))

print(f"Exact byte-identical images across train/val: {len(exact_byte_dupes)}")

# ---------- 3. NEAR-DUPLICATE CHECK (PERCEPTUAL HASH) ----------
def phash_of(path, hash_size=16):
    try:
        img = Image.open(path).convert("RGB")
        return imagehash.phash(img, hash_size=hash_size)
    except Exception:
        return None

print("\nComputing perceptual hashes (this can take a while for large datasets)...")
train_phash = {p: phash_of(p) for p in train_paths}
val_phash = {p: phash_of(p) for p in val_paths}

train_phash = {p: h for p, h in train_phash.items() if h is not None}
val_phash = {p: h for p, h in val_phash.items() if h is not None}

HAMMING_THRESHOLD = 5  # <=5 bits diff = visually near-identical

near_dupes = []
train_items = list(train_phash.items())
for vp, vh in val_phash.items():
    for tp, th in train_items:
        if (vh - th) <= HAMMING_THRESHOLD:
            near_dupes.append((tp, vp, vh - th))

near_dupes_df = pd.DataFrame(near_dupes, columns=["train_path", "val_path", "hamming_distance"])
near_dupes_df.to_csv(f"{OUTPUT_DIR}/leakage_near_duplicates.csv", index=False)

print(f"Near-duplicate image pairs (hamming <= {HAMMING_THRESHOLD}): {len(near_dupes)}")
if len(near_dupes) > 0:
    affected_val = near_dupes_df["val_path"].nunique()
    print(f"Affected val images: {affected_val} / {len(val_paths)} "
          f"({100*affected_val/len(val_paths):.2f}%)")

# ---------- 4. CLASS-LEVEL LEAKAGE SUMMARY ----------
summary = {
    "n_train": len(train_paths),
    "n_val": len(val_paths),
    "exact_path_overlap": len(exact_path_overlap),
    "filename_overlap": len(fname_overlap),
    "exact_byte_duplicates": len(exact_byte_dupes),
    "near_duplicate_pairs": len(near_dupes),
    "near_duplicate_affected_val_pct": round(
        100 * near_dupes_df["val_path"].nunique() / len(val_paths), 2
    ) if len(near_dupes) > 0 else 0.0,
}
pd.DataFrame([summary]).to_csv(f"{OUTPUT_DIR}/leakage_summary.csv", index=False)

print("\n======================")
print("LEAKAGE CHECK SUMMARY")
print("======================")
for k, v in summary.items():
    print(f"{k}: {v}")

if summary["exact_byte_duplicates"] == 0 and summary["near_duplicate_pairs"] == 0:
    print("\n✅ No train/val leakage detected.")
else:
    print("\n⚠️ Potential leakage found — review leakage_near_duplicates.csv before reporting results.")

In [ ]:
# ==================================================================================
# RoseSSL-CBAMNet — ADVANCED ANALYSIS SUITE



import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from scipy import stats

print("TensorFlow:", tf.__version__)

# ========================== CONFIG ==========================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"
SSL_ENCODER_PATH = "ssl_encoder.keras"     # produced once by your existing SimCLR script
CLASSIFIER_EPOCHS = 30                     # matches the paper's stated max (fix vs. old CLASSIFIER_EPOCHS=10)
FIXED_TEST_SEED = 42                       # test split is carved ONCE with this seed and never reshuffled
SEEDS = [42, 123, 2024, 7, 1337]           # multi-seed pool for variance reporting + paired ablation
OUTPUT_DIR = "advanced_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUTOTUNE = tf.data.AUTOTUNE

# ========================== LOAD FILE LIST (once) ==========================
class_names = sorted(os.listdir(COMBINED_PATH))
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

image_paths, labels = [], []
for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in os.listdir(folder):
        image_paths.append(os.path.join(folder, img))
        labels.append(class_to_idx[cls])

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Classes:", NUM_CLASSES, class_names)
print("Total images:", len(image_paths))

# ========================== FIXED HELD-OUT TEST SET (carved ONCE) ==========================
trainpool_paths, test_paths, trainpool_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.20,
    random_state=FIXED_TEST_SEED, stratify=labels
)
print(f"Train pool: {len(trainpool_paths)}  |  Held-out TEST (fixed, seed={FIXED_TEST_SEED}): {len(test_paths)}")
print("This test set is identical for every seed/config below — it is never used for model selection.")

# ========================== DATA PIPELINE HELPERS ==========================
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

def make_dataset(paths, labs, shuffle=False, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((paths, labs))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=seed)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

test_ds = make_dataset(test_paths, test_labels, shuffle=False)

# ========================== CBAM (unchanged from original) ==========================
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)

    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)

    shared_dense_1 = layers.Dense(channel // ratio, activation="relu")
    shared_dense_2 = layers.Dense(channel)

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Add()([avg_out, max_out])
    channel_attention = layers.Activation("sigmoid")(channel_attention)
    channel_refined = layers.Multiply()([input_feature, channel_attention])

    avg_pool_spatial = ChannelMean()(channel_refined)
    max_pool_spatial = ChannelMax()(channel_refined)
    concat = layers.Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])

    spatial_attention = layers.Conv2D(
        filters=1, kernel_size=7, padding="same", activation="sigmoid"
    )(concat)

    refined_feature = layers.Multiply()([channel_refined, spatial_attention])
    return refined_feature

# ========================== MODEL BUILDER (parametrized for ablation) ==========================
def build_classifier(use_ssl_encoder=True, use_cbam=True, seed=42):
    """
    use_ssl_encoder=False, use_cbam=False -> "MobileNetV2" baseline row
    use_ssl_encoder=True,  use_cbam=False -> "SimCLR + MobileNetV2" row
    use_ssl_encoder=True,  use_cbam=True  -> "SimCLR + MobileNetV2 + CBAM" (proposed / Full)
    """
    tf.keras.utils.set_random_seed(seed)  # seeds Python, NumPy, and TF RNGs together

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

    if use_ssl_encoder:
        backbone = tf.keras.models.load_model(SSL_ENCODER_PATH)
    else:
        backbone = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    x = backbone(x, training=False)

    if use_cbam:
        x = cbam_block(x, ratio=8)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ========================== TRAIN + EVALUATE ONE (config, seed) PAIR ==========================
def train_and_evaluate(config_name, use_ssl_encoder, use_cbam, seed, verbose=0):
    """
    Splits the TRAIN POOL only (never the fixed test set) into train/val using `seed`,
    trains the model, then evaluates ONCE on the fixed, untouched test_ds/test_labels.
    """
    tr_paths, va_paths, tr_labels, va_labels = train_test_split(
        trainpool_paths, trainpool_labels,
        test_size=0.11,            # ~0.11 of the 80% pool -> ~71/9/20 overall train/val/test
        random_state=seed, stratify=trainpool_labels
    )

    train_ds = make_dataset(tr_paths, tr_labels, shuffle=True, seed=seed)
    val_ds = make_dataset(va_paths, va_labels, shuffle=False)

    model = build_classifier(use_ssl_encoder=use_ssl_encoder, use_cbam=use_cbam, seed=seed)

    ckpt_path = os.path.join(OUTPUT_DIR, f"{config_name}_seed{seed}.keras")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=0
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=3, verbose=0
        ),
    ]

    model.fit(
        train_ds, validation_data=val_ds,
        epochs=CLASSIFIER_EPOCHS, callbacks=callbacks, verbose=verbose
    )

    # ---- Evaluate ONCE on the fixed, held-out test set (never seen during training/selection) ----
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_labels

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    y_true_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    macro_auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")

    del model
    tf.keras.backend.clear_session()

    return {
        "config": config_name, "seed": seed,
        "accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
        "macro_auc": macro_auc,
        "y_true": y_true, "y_prob": y_prob
    }

# ==================================================================================
# PART A: MULTI-SEED EVALUATION + VARIANCE REPORTING (full proposed model)
# ==================================================================================
print("\n" + "=" * 78)
print("PART A: MULTI-SEED EVALUATION — SimCLR + MobileNetV2 + CBAM (Full model)")
print("=" * 78)

multiseed_results = []
for seed in SEEDS:
    print(f"\n--- Training FULL model | seed={seed} ---")
    res = train_and_evaluate("Full_SimCLR_CBAM", use_ssl_encoder=True, use_cbam=True,
                              seed=seed, verbose=1)
    multiseed_results.append(res)
    print(f"seed={seed} -> Acc={res['accuracy']:.4f}  F1={res['f1']:.4f}  "
          f"MacroAUC={res['macro_auc']:.4f}")

ms_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("y_true", "y_prob")}
                       for r in multiseed_results])
ms_df.to_csv(os.path.join(OUTPUT_DIR, "multiseed_per_seed_results.csv"), index=False)

summary_rows = []
for metric in ["accuracy", "precision", "recall", "f1", "macro_auc"]:
    vals = ms_df[metric].values
    mean, std = vals.mean(), vals.std(ddof=1)
    cv = 100 * std / mean
    summary_rows.append({
        "Metric": metric, "Mean": mean, "Std": std, "CV(%)": cv,
        "Min": vals.min(), "Max": vals.max(), "Range": vals.max() - vals.min()
    })
ms_summary = pd.DataFrame(summary_rows)
ms_summary.to_csv(os.path.join(OUTPUT_DIR, "multiseed_summary.csv"), index=False)

print("\nMulti-seed summary (report this table in the paper as mean ± std, CV%):")
print(ms_summary.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.bar(range(len(SEEDS)), ms_df["macro_auc"], color="#4C72B0")
plt.axhline(ms_df["macro_auc"].mean(), color="red", linestyle="--",
            label=f"Mean = {ms_df['macro_auc'].mean():.4f}")
plt.xticks(range(len(SEEDS)), [f"seed {s}" for s in SEEDS])
plt.ylabel("Macro AUC (held-out test set)")
plt.title("Per-Seed AUC Dispersion — Full RoseSSL-CBAMNet")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "multiseed_auc_dispersion.png"), dpi=600)
plt.show()

# ==================================================================================
# SUMMARY OF SAVED FILES
# ==================================================================================
print("\n" + "=" * 78)
print(f"All outputs saved under: {OUTPUT_DIR}/")
print("=" * 78)
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith((".csv", ".png")):
        print(" -", f)

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CKPT_YTRUE = os.path.join(OUTPUT_DIR, "threshold_y_true.npy")
CKPT_YPROB = os.path.join(OUTPUT_DIR, "threshold_y_prob.npy")
CKPT_CLASSNAMES = os.path.join(OUTPUT_DIR, "threshold_class_names.npy")

# ---- safety: reload from checkpoint if kernel was restarted ----
try:
    y_true
    y_prob
    class_names
except NameError:
    y_true = np.load(CKPT_YTRUE)
    y_prob = np.load(CKPT_YPROB)
    class_names = list(np.load(CKPT_CLASSNAMES, allow_pickle=True))
    print("Reloaded y_true / y_prob / class_names from checkpoint")

# ---- always save checkpoint right away so future reruns are safe ----
np.save(CKPT_YTRUE, np.asarray(y_true))
np.save(CKPT_YPROB, np.asarray(y_prob))
np.save(CKPT_CLASSNAMES, np.asarray(class_names, dtype=object))

n_classes = len(class_names)
y_true_arr = np.asarray(y_true)
y_prob_arr = np.asarray(y_prob)

THRESHOLDS = np.linspace(0.001, 0.999, 499)   # finer grid for smooth curves

results = {}          # per-class curves
summary_records = []  # per-class summary row

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.ravel()

for i, cls in enumerate(class_names):
    y_bin = (y_true_arr == i).astype(int)
    scores = y_prob_arr[:, i]
    n_pos = int(y_bin.sum())

    fpr, tpr, _ = roc_curve(y_bin, scores)
    roc_auc = auc(fpr, tpr)

    sens_list, spec_list, acc_list = [], [], []
    for t in THRESHOLDS:
        y_hat = (scores >= t).astype(int)
        tp = np.sum((y_hat == 1) & (y_bin == 1))
        tn = np.sum((y_hat == 0) & (y_bin == 0))
        fp = np.sum((y_hat == 1) & (y_bin == 0))
        fn = np.sum((y_hat == 0) & (y_bin == 1))

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        acc = (tp + tn) / len(y_bin)

        sens_list.append(sens)
        spec_list.append(spec)
        acc_list.append(acc)

    sens_arr = np.array(sens_list)
    spec_arr = np.array(spec_list)
    acc_arr = np.array(acc_list)
    diff_arr = sens_arr - spec_arr

    # accuracy-optimal threshold
    t_acc_idx = np.argmax(acc_arr)
    t_acc = THRESHOLDS[t_acc_idx]

    # sensitivity == specificity: interpolate the true zero-crossing of diff_arr
    sign_change = np.where(np.diff(np.sign(diff_arr)) != 0)[0]
    if len(sign_change) > 0:
        j = sign_change[np.argmin(np.abs(THRESHOLDS[sign_change] - 0.5))]
        x0, x1 = THRESHOLDS[j], THRESHOLDS[j + 1]
        y0, y1 = diff_arr[j], diff_arr[j + 1]
        t_bal = x0 - y0 * (x1 - x0) / (y1 - y0) if (y1 - y0) != 0 else x0
        t_bal = float(np.clip(t_bal, THRESHOLDS.min(), THRESHOLDS.max()))
        t_bal_idx = int(np.argmin(np.abs(THRESHOLDS - t_bal)))
    else:
        t_bal_idx = int(np.argmin(np.abs(diff_arr)))
        t_bal = THRESHOLDS[t_bal_idx]

    # default t=0.50 (closest grid point)
    t50_idx = int(np.argmin(np.abs(THRESHOLDS - 0.50)))

    results[cls] = dict(
        thresholds=THRESHOLDS, sens=sens_arr, spec=spec_arr, acc=acc_arr,
        auc=roc_auc, t_acc=t_acc, t_bal=t_bal, n_pos=n_pos
    )

    summary_records.append({
        "Class": cls,
        "AUC": roc_auc,
        "n_pos": n_pos,
        "Sens_at_0.50": sens_arr[t50_idx],
        "Spec_at_0.50": spec_arr[t50_idx],
        "SensMinusSpec_at_0.50": sens_arr[t50_idx] - spec_arr[t50_idx],
        "t_acc": t_acc,
        "Sens_at_t_acc": sens_arr[t_acc_idx],
        "Spec_at_t_acc": spec_arr[t_acc_idx],
        "Accuracy_at_t_acc": acc_arr[t_acc_idx],
        "t_bal": t_bal,
        "Sens_at_t_bal": sens_arr[t_bal_idx],
        "Spec_at_t_bal": spec_arr[t_bal_idx],
    })

    # ---- plot (shared-legend, shaded-region style) ----
    ax = axes[i]

    ax.fill_between(THRESHOLDS, sens_arr, spec_arr, where=(sens_arr >= spec_arr),
                     color="firebrick", alpha=0.12, interpolate=True)
    ax.fill_between(THRESHOLDS, sens_arr, spec_arr, where=(sens_arr < spec_arr),
                     color="steelblue", alpha=0.12, interpolate=True)

    l_sens, = ax.plot(THRESHOLDS, sens_arr, color="firebrick", lw=1.8, label="Sensitivity")
    l_spec, = ax.plot(THRESHOLDS, spec_arr, color="steelblue", lw=1.8, label="Specificity")
    l_acc, = ax.plot(THRESHOLDS, acc_arr, color="mediumpurple", lw=1.3, linestyle="--", label="Accuracy")

    l_t50 = ax.axvline(0.50, color="gray", linestyle=":", lw=1.2, label="t=0.50 (default)")
    l_tacc = ax.axvline(t_acc, color="black", linestyle="-", lw=1.0,
                         label=r"$t_{acc}$ (accuracy-optimal)")
    l_tbal = ax.axvline(t_bal, color="dimgray", linestyle="--", lw=1.0,
                         label=r"$t_{bal}$ (sens = spec)")

    # circle markers on each curve at t_acc
    ax.plot(t_acc, sens_arr[t_acc_idx], "o", color="firebrick", ms=6, zorder=5)
    ax.plot(t_acc, spec_arr[t_acc_idx], "o", color="steelblue", ms=6, zorder=5)
    # square marker at the true sens=spec crossover
    ax.plot(t_bal, sens_arr[t_bal_idx], "s", color="white", mec="black", mew=1.3, ms=6, zorder=6)

    diff_050 = sens_arr[t50_idx] - spec_arr[t50_idx]
    diff_tacc = sens_arr[t_acc_idx] - spec_arr[t_acc_idx]
    diff_tbal = sens_arr[t_bal_idx] - spec_arr[t_bal_idx]
    textbox = (
        f"t=0.50 (default):      sens - spec = {diff_050:+.3f}\n"
        f"t_acc={t_acc:.2f} (acc.-opt):   sens - spec = {diff_tacc:+.3f}\n"
        f"t_bal={t_bal:.2f} (balanced):   sens - spec = {diff_tbal:+.3f}"
    )
    ax.text(0.02, 0.03, textbox, transform=ax.transAxes, fontsize=8,
            va="bottom", ha="left", family="monospace",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.9))

    ax.set_title(f"{cls}  (AUC = {roc_auc:.3f},  n = {n_pos})", fontsize=12)
    ax.set_xlabel("Decision threshold t")
    ax.set_ylabel("Operating-point metric")
    ax.set_xlim(0, 1)
    y_min = min(sens_arr.min(), spec_arr.min(), acc_arr.min())
    ax.set_ylim(max(0, y_min - 0.03), 1.01)
    ax.grid(alpha=0.15)

# ---- single shared legend at the top of the figure ----
handles = [l_sens, l_spec, l_acc, l_t50, l_tacc, l_tbal]
labels = [h.get_label() for h in handles]
fig.legend(handles, labels, loc="upper center", ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, 1.03), frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUTPUT_DIR, "sabbir_threshold_sensitivity_all_classes.png"),
            dpi=600, bbox_inches="tight")
plt.show()

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "threshold_sensitivity_summary.csv"), index=False)
print(summary_df.to_string(index=False))

# ============================================================
# APPLY CALIBRATED PER-CLASS THRESHOLDS (t_acc) — fixes the
# sensitivity drop at the default 0.50 cutoff, per-class.
#
# NOTE: an earlier "exactly-one-class-passes" rule was a no-op here,
# because Insect Hole's t_acc (0.01) is so low that almost every
# sample clears it -> almost every sample had >1 class "passing" ->
# the fallback (plain argmax) fired on ~100% of samples, so the
# calibrated result was identical to the uncalibrated one.
#
# Correct rule: among the classes that clear THEIR OWN t_acc, pick
# the one with the highest probability. This can override the global
# argmax when the argmax class itself doesn't clear its own bar
# (an overconfident/miscalibrated call) in favor of another class
# that does. Only fall back to plain argmax when NO class clears
# its threshold at all.
# ============================================================
t_acc_per_class = np.array([results[cls]["t_acc"] for cls in class_names])

passed = y_prob_arr >= t_acc_per_class[np.newaxis, :]     # (N, n_classes) bool
n_passed = passed.sum(axis=1)

masked_probs = np.where(passed, y_prob_arr, -np.inf)       # -inf where not passed
y_pred_calibrated = np.argmax(masked_probs, axis=1)

# no class passed -> fall back to plain argmax of raw probabilities
none_passed_mask = n_passed == 0
y_pred_calibrated[none_passed_mask] = np.argmax(y_prob_arr[none_passed_mask], axis=1)

n_overridden = np.sum(y_pred_calibrated != np.argmax(y_prob_arr, axis=1))
print(f"Predictions changed vs plain argmax: {n_overridden} / {len(y_true_arr)}")

from sklearn.metrics import classification_report, accuracy_score

calibrated_acc = accuracy_score(y_true_arr, y_pred_calibrated)
default_acc = accuracy_score(y_true_arr, np.argmax(y_prob_arr, axis=1))

print(f"\nDefault argmax accuracy      : {default_acc:.4f}")
print(f"Calibrated-threshold accuracy: {calibrated_acc:.4f}")
print("\nCalibrated-threshold classification report:")
calibrated_report = classification_report(y_true_arr, y_pred_calibrated, target_names=class_names, digits=4)
print(calibrated_report)

with open(os.path.join(OUTPUT_DIR, "calibrated_threshold_classification_report.txt"), "w") as f:
    f.write(f"Per-class t_acc thresholds: {dict(zip(class_names, t_acc_per_class))}\n\n")
    f.write(calibrated_report)

print("\nSaved Files:")
print("threshold_sensitivity_all_classes.png")
print("threshold_sensitivity_summary.csv")
print("calibrated_threshold_classification_report.txt")

In [ ]:

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, silhouette_score,
    davies_bouldin_score
)
from sklearn.manifold import TSNE

try:
    import cv2
except ImportError:
    raise ImportError("This script needs OpenCV. Run: pip install opencv-python --break-system-packages")

print("TensorFlow:", tf.__version__)

# ========================== CONFIG ==========================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"
SSL_ENCODER_PATH = "ssl_encoder.keras"                 # from Part-1 SimCLR script
FULL_MODEL_PATH = "AgriSSL_CBAMNet_labdata.keras"       # your trained SimCLR+CBAM classifier
FIXED_TEST_SEED = 42
OUTPUT_DIR = "robustness_embedding_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

.
CROSS_DOMAIN_PATH = None   

RUN_PGD = False   

AUTOTUNE = tf.data.AUTOTUNE

# ========================== FILE LIST + FIXED TEST SPLIT (same convention as before) ==========================
class_names = sorted(os.listdir(COMBINED_PATH))
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

image_paths, labels = [], []
for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in os.listdir(folder):
        image_paths.append(os.path.join(folder, img))
        labels.append(class_to_idx[cls])
image_paths = np.array(image_paths)
labels = np.array(labels)

trainpool_paths, test_paths, trainpool_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.20, random_state=FIXED_TEST_SEED, stratify=labels
)
print(f"Held-out TEST set (fixed, seed={FIXED_TEST_SEED}): {len(test_paths)} images")

# ========================== LOAD TEST SET INTO MEMORY (needed for pixel-level corruption/attack) ==========================
def load_image_array(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.numpy().astype(np.float32)   # kept in 0-255 range; model applies preprocess_input internally

print("Loading held-out test images into memory for corruption/attack experiments...")
X_test = np.stack([load_image_array(p) for p in test_paths]).astype(np.float32)
y_test = test_labels.copy()
print("X_test shape:", X_test.shape)

def predict_in_batches(model, images_array, batch_size=BATCH_SIZE):
    probs = []
    for start in range(0, len(images_array), batch_size):
        batch = images_array[start:start + batch_size]
        probs.append(model.predict(batch, verbose=0))
    return np.concatenate(probs, axis=0)

# ========================== CBAM (unchanged) ==========================
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.Reshape((1, 1, channel))(layers.GlobalAveragePooling2D()(input_feature))
    max_pool = layers.Reshape((1, 1, channel))(layers.GlobalMaxPooling2D()(input_feature))
    shared_dense_1 = layers.Dense(channel // ratio, activation="relu")
    shared_dense_2 = layers.Dense(channel)
    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))
    channel_attention = layers.Activation("sigmoid")(layers.Add()([avg_out, max_out]))
    channel_refined = layers.Multiply()([input_feature, channel_attention])
    concat = layers.Concatenate(axis=-1)([ChannelMean()(channel_refined), ChannelMax()(channel_refined)])
    spatial_attention = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(concat)
    return layers.Multiply()([channel_refined, spatial_attention])

# ========================== LOAD THE TRAINED FULL MODEL ==========================
if not os.path.exists(FULL_MODEL_PATH):
    raise FileNotFoundError(
        f"Could not find '{FULL_MODEL_PATH}'. Point FULL_MODEL_PATH at your trained "
        f"SimCLR+CBAM classifier (.keras file) before running this script."
    )
full_model = tf.keras.models.load_model(
    FULL_MODEL_PATH, custom_objects={"ChannelMean": ChannelMean, "ChannelMax": ChannelMax}
)
print("Loaded full trained model:", FULL_MODEL_PATH)

# Clean baseline (severity 0) — everything else is measured as a drop from this
y_prob_clean = predict_in_batches(full_model, X_test)
y_pred_clean = np.argmax(y_prob_clean, axis=1)
acc_clean = accuracy_score(y_test, y_pred_clean)
_, _, f1_clean, _ = precision_recall_fscore_support(y_test, y_pred_clean, average="weighted", zero_division=0)
print(f"Clean test accuracy: {acc_clean:.4f}  |  Clean weighted F1: {f1_clean:.4f}")

# ==================================================================================
# PART D: NOISE / CORRUPTION ROBUSTNESS
# ==================================================================================
print("\n" + "=" * 78)
print("PART D: ROBUSTNESS UNDER IMAGE PERTURBATIONS")
print("=" * 78)

def apply_gaussian_noise(images, sigma):
    if sigma == 0:
        return images
    noise = np.random.normal(0, sigma, images.shape).astype(np.float32)
    return np.clip(images + noise, 0, 255)

def apply_gaussian_blur(images, ksize):
    if ksize <= 1:
        return images
    out = np.empty_like(images)
    for i in range(images.shape[0]):
        out[i] = cv2.GaussianBlur(images[i], (ksize, ksize), 0)
    return out

def apply_brightness(images, factor):
    if factor == 1.0:
        return images
    return np.clip(images * factor, 0, 255)

def apply_jpeg_compression(images, quality):
    if quality >= 100:
        return images
    out = np.empty_like(images)
    params = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    for i in range(images.shape[0]):
        bgr = cv2.cvtColor(images[i].astype(np.uint8), cv2.COLOR_RGB2BGR)
        _, enc = cv2.imencode(".jpg", bgr, params)
        dec_bgr = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        out[i] = cv2.cvtColor(dec_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    return out

CORRUPTIONS = {
    "Gaussian Noise (sigma)": (apply_gaussian_noise, [0, 10, 20, 30]),
    "Gaussian Blur (kernel)": (apply_gaussian_blur, [1, 3, 5, 7]),
    "Brightness (factor)":    (apply_brightness, [1.0, 1.3, 0.7, 0.5]),
    "JPEG Compression (quality)": (apply_jpeg_compression, [100, 50, 30, 10]),
}

robustness_rows = []
for corruption_name, (fn, severities) in CORRUPTIONS.items():
    for sev in severities:
        X_corrupt = fn(X_test.copy(), sev)
        y_prob = predict_in_batches(full_model, X_corrupt)
        y_pred = np.argmax(y_prob, axis=1)
        acc = accuracy_score(y_test, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
        robustness_rows.append({
            "Corruption": corruption_name, "Severity": sev,
            "Accuracy": acc, "F1": f1,
            "Accuracy Drop (pp)": (acc_clean - acc) * 100
        })
        print(f"{corruption_name:28s} severity={sev!s:>6} -> Acc={acc:.4f}  (drop={100*(acc_clean-acc):.2f} pp)")

robust_df = pd.DataFrame(robustness_rows)
robust_df.to_csv(os.path.join(OUTPUT_DIR, "robustness_perturbation_results.csv"), index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
for i, (corruption_name, (_, severities)) in enumerate(CORRUPTIONS.items()):
    sub = robust_df[robust_df["Corruption"] == corruption_name]
    axes[i].plot(range(len(severities)), sub["Accuracy"], marker="o", color="#4C72B0")
    axes[i].axhline(acc_clean, color="gray", linestyle="--", label="Clean accuracy")
    axes[i].set_xticks(range(len(severities)))
    axes[i].set_xticklabels(severities)
    axes[i].set_title(corruption_name)
    axes[i].set_xlabel("Severity")
    axes[i].set_ylabel("Accuracy")
    axes[i].legend(fontsize=8)
plt.suptitle("Robustness Analysis under Image Perturbations")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "robustness_perturbation_curves.png"), dpi=600)
plt.show()

print("\nRobustness summary table (paste into paper as 'Table: Robustness under Image Perturbations'):")
print(robust_df.to_string(index=False))



# ==================================================================================
# SUMMARY OF SAVED FILES
# ==================================================================================
print("\n" + "=" * 78)
print(f"All outputs saved under: {OUTPUT_DIR}/")
print("=" * 78)
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith((".csv", ".png")):
        print(" -", f)

In [ ]:
# ============================================================
# GRAD-CAM & GRAD-CAM++ IMPLEMENTATION
# ============================================================
import cv2
import os

# Safety: redefine custom layers + reload model if kernel was restarted
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

try:
    model
except NameError:
    model = tf.keras.models.load_model(
        "/kaggle/working/best_model.keras",
        custom_objects={"ChannelMean": ChannelMean, "ChannelMax": ChannelMax}
    )
    print("Model reloaded from checkpoint")

# ========================== AUTO-DETECT TARGET CONV LAYER ==========================
# Last spatial (4D) feature map before GlobalAveragePooling -> CBAM refined feature.
# NOTE (Keras 3): plain layers no longer expose `.output_shape`, only nested
# Model instances (like the MobileNetV2 submodel) do -> using output_shape would
# wrongly select the raw encoder instead of the CBAM-refined feature, and
# extracting an intermediate tensor from a nested submodel breaks graph tracing
# (KeyError). So: use `.output.shape` and explicitly skip nested Model layers.
def get_last_spatial_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.Model):
            continue  # skip nested submodels (e.g. the MobileNetV2 encoder)
        try:
            out = layer.output
        except Exception:
            continue
        out_shape = out.shape if not isinstance(out, list) else out[0].shape
        if out_shape is not None and len(out_shape) == 4:
            return layer
    raise ValueError("No 4D spatial layer found in model")

TARGET_LAYER = get_last_spatial_layer(model)
print("Grad-CAM target layer:", TARGET_LAYER.name, TARGET_LAYER.output.shape)

GRAD_MODEL = tf.keras.models.Model(
    inputs=model.input,
    outputs=[TARGET_LAYER.output, model.output]
)

# ========================== GRAD-CAM ==========================
def compute_gradcam(img_array, pred_index=None):
    with tf.GradientTape() as tape:
        conv_output, preds = GRAD_MODEL(img_array, training=False)
        preds = preds[0]
        if pred_index is None:
            pred_index = int(tf.argmax(preds))
        class_channel = tf.expand_dims(preds[pred_index], axis=0)

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]

    heatmap = tf.reduce_sum(conv_output * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    confidence = float(preds[pred_index].numpy())
    return heatmap.numpy(), pred_index, confidence

# ========================== GRAD-CAM++ ==========================
def compute_gradcam_plusplus(img_array, pred_index=None):
    with tf.GradientTape() as tape:
        conv_output, preds = GRAD_MODEL(img_array, training=False)
        preds = preds[0]
        if pred_index is None:
            pred_index = int(tf.argmax(preds))
        class_channel = tf.expand_dims(preds[pred_index], axis=0)

    grads = tape.gradient(class_channel, conv_output)
    conv_output = conv_output[0]
    grads = grads[0]

    grads_sq = tf.square(grads)
    grads_cube = grads_sq * grads

    sum_A = tf.reduce_sum(conv_output, axis=(0, 1), keepdims=True)
    alpha_denom = 2.0 * grads_sq + sum_A * grads_cube
    alpha_denom = tf.where(alpha_denom != 0, alpha_denom, tf.ones_like(alpha_denom) * 1e-8)
    alphas = grads_sq / alpha_denom

    weights = tf.reduce_sum(alphas * tf.nn.relu(grads), axis=(0, 1))
    heatmap = tf.reduce_sum(conv_output * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    confidence = float(preds[pred_index].numpy())
    return heatmap.numpy(), pred_index, confidence

# ========================== OVERLAY HELPER ==========================
def overlay_heatmap(img_float, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img_float.shape[1], img_float.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = heatmap_color.astype(np.float32) * alpha + img_float
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return overlay, heatmap_resized

# ========================== SAMPLE VISUALIZATION (ALL CLASSES) ==========================
# ager code e prothom N ta val image niche felto (sequential index) -> jodi val set
# order e shob class thake na ba ekhane 2 class dominant thake, তাহলে shudhu 1-2 class e
# dekha jeto. Ekhon protita class theke explicitly n_samples ta kore image select kora hocche.
SAMPLES_PER_CLASS = 1  # protita class theke koyta shomple dekhte chao, dorkar hole barao

val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

n_rows = len(selected_indices)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for row, idx in enumerate(selected_indices):
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()
    img_batch = img_np[np.newaxis, ...]

    heatmap_gc, pred_idx, conf_gc = compute_gradcam(img_batch)
    overlay_gc, _ = overlay_heatmap(img_np, heatmap_gc)

    heatmap_gcpp, _, conf_gcpp = compute_gradcam_plusplus(img_batch, pred_index=pred_idx)
    overlay_gcpp, _ = overlay_heatmap(img_np, heatmap_gcpp)

    axes[row, 0].imshow(img_np.astype(np.uint8))
    axes[row, 0].set_title(f"Original\nTrue: {class_names[true_label]}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay_gc)
    axes[row, 1].set_title(f"Grad-CAM\nPred: {class_names[pred_idx]} ({conf_gc*100:.2f}%)")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay_gcpp)
    axes[row, 2].set_title(f"Grad-CAM++\nPred: {class_names[pred_idx]} ({conf_gcpp*100:.2f}%)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/gradcam_vs_gradcampp.png", dpi=600)
plt.show()
print("Saved: gradcam_vs_gradcampp.png (all", len(class_names), "classes covered)")


In [ ]:
# ============================================================
# INSERTION / DELETION AUC (Grad-CAM & Grad-CAM++)
# Patch-based causal faithfulness metric (Petsiuk et al., RISE)
# Session-resumable: safe against Kaggle kernel resets
# ============================================================
CHECKPOINT_PATH = "/kaggle/working/xai_causal_metrics_progress.csv"
SUMMARY_PATH = "/kaggle/working/xai_causal_metrics_summary.csv"

N_EVAL_SAMPLES = min(150, len(val_paths))   # bariye/komiye dite paro time budget onujayi
PATCH_SIZE = 16
N_STEPS = 20

def get_patch_order(heatmap_224, patch_size=16):
    h, w = heatmap_224.shape
    n_h, n_w = h // patch_size, w // patch_size
    patch_scores = np.zeros((n_h, n_w))
    for i in range(n_h):
        for j in range(n_w):
            patch_scores[i, j] = heatmap_224[
                i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size
            ].mean()
    flat_order = np.argsort(-patch_scores.ravel())
    order = np.stack(np.unravel_index(flat_order, patch_scores.shape), axis=1)
    return order

def causal_metric(img_224, heatmap_224, pred_index, mode="insertion",
                   patch_size=16, n_steps=20):
    order = get_patch_order(heatmap_224, patch_size)
    n_patches = len(order)
    step = max(1, n_patches // n_steps)

    blurred = cv2.GaussianBlur(img_224, (31, 31), 0)
    canvas = blurred.copy() if mode == "insertion" else img_224.copy()

    scores, fractions = [], []
    for idx in range(0, n_patches + step, step):
        idx = min(idx, n_patches)
        for (pi, pj) in order[:idx]:
            r0, r1 = pi*patch_size, (pi+1)*patch_size
            c0, c1 = pj*patch_size, (pj+1)*patch_size
            if mode == "insertion":
                canvas[r0:r1, c0:c1] = img_224[r0:r1, c0:c1]
            else:
                canvas[r0:r1, c0:c1] = blurred[r0:r1, c0:c1]

        pred = model.predict(canvas[np.newaxis, ...], verbose=0)[0]
        scores.append(pred[pred_index])
        fractions.append(idx / n_patches)
        if idx == n_patches:
            break

    return float(np.trapz(scores, fractions))

# ========================== RESUME FROM CHECKPOINT ==========================
if os.path.exists(CHECKPOINT_PATH):
    results_df = pd.read_csv(CHECKPOINT_PATH)
    done_indices = set(results_df["image_index"].tolist())
    print(f"Resuming... {len(done_indices)}/{N_EVAL_SAMPLES} images already done")
else:
    results_df = pd.DataFrame(columns=[
        "image_index", "true_label", "pred_label", "confidence",
        "GradCAM_Insertion_AUC", "GradCAM_Deletion_AUC",
        "GradCAM++_Insertion_AUC", "GradCAM++_Deletion_AUC"
    ])
    done_indices = set()

eval_paths = val_paths[:N_EVAL_SAMPLES]
eval_labels = val_labels[:N_EVAL_SAMPLES]

for idx, (path, true_label) in enumerate(zip(eval_paths, eval_labels)):
    if idx in done_indices:
        continue

    img, _ = load_image(path, true_label)
    img_np = img.numpy()
    img_batch = img_np[np.newaxis, ...]

    heatmap_gc, pred_idx, confidence = compute_gradcam(img_batch)
    heatmap_gc_224 = cv2.resize(heatmap_gc, (IMG_SIZE, IMG_SIZE))

    heatmap_gcpp, _, _ = compute_gradcam_plusplus(img_batch, pred_index=pred_idx)
    heatmap_gcpp_224 = cv2.resize(heatmap_gcpp, (IMG_SIZE, IMG_SIZE))

    ins_gc = causal_metric(img_np, heatmap_gc_224, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_gc = causal_metric(img_np, heatmap_gc_224, pred_idx, "deletion", PATCH_SIZE, N_STEPS)
    ins_gcpp = causal_metric(img_np, heatmap_gcpp_224, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_gcpp = causal_metric(img_np, heatmap_gcpp_224, pred_idx, "deletion", PATCH_SIZE, N_STEPS)

    results_df.loc[len(results_df)] = [
        idx, true_label, pred_idx, confidence, ins_gc, del_gc, ins_gcpp, del_gcpp
    ]

    if idx % 10 == 0:
        results_df.to_csv(CHECKPOINT_PATH, index=False)
        print(f"[{idx+1}/{N_EVAL_SAMPLES}] checkpoint saved")

results_df.to_csv(CHECKPOINT_PATH, index=False)

# ========================== SUMMARY ==========================
summary = pd.DataFrame({
    "Mean_Confidence":         [results_df["confidence"].mean()],
    "GradCAM_Insertion_AUC":   [results_df["GradCAM_Insertion_AUC"].mean()],
    "GradCAM_Deletion_AUC":    [results_df["GradCAM_Deletion_AUC"].mean()],
    "GradCAM++_Insertion_AUC": [results_df["GradCAM++_Insertion_AUC"].mean()],
    "GradCAM++_Deletion_AUC":  [results_df["GradCAM++_Deletion_AUC"].mean()],
})
summary.to_csv(SUMMARY_PATH, index=False)

print("\n======================")
print("XAI Causal Faithfulness Metrics (mean over", len(results_df), "images)")
print("======================")
print(summary.T)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

SAMPLES_PER_CLASS = 1  # dorkar hole barao
val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

lime_explainer = lime_image.LimeImageExplainer()

LIME_NUM_SAMPLES = 1000
LIME_NUM_FEATURES = 8

n_rows = len(selected_indices)
n_cols = len(class_names) + 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

lime_records = []

for row, idx in enumerate(selected_indices):
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()

    preds = predict_fn(img_np[np.newaxis, ...])[0]
    pred_idx = int(np.argmax(preds))

    explanation = lime_explainer.explain_instance(
        img_np.astype(np.double),
        predict_fn,
        top_labels=len(class_names),
        hide_color=0,
        num_samples=LIME_NUM_SAMPLES
    )

    axes[row, 0].imshow(img_np.astype(np.uint8))
    axes[row, 0].set_title(
        f"Original\nTrue: {class_names[true_label]}\n"
        f"Pred: {class_names[pred_idx]} ({preds[pred_idx]*100:.2f}%)"
    )
    axes[row, 0].axis("off")

    for c in range(len(class_names)):
        temp, mask = explanation.get_image_and_mask(
            c, positive_only=True, num_features=LIME_NUM_FEATURES, hide_rest=False
        )
        vis = mark_boundaries(np.clip(temp, 0, 255) / 255.0, mask)

        # LIME's own metric: R^2 fidelity of the local linear surrogate model
        # (how well the simple local model approximates the black-box model
        #  in the neighborhood of this instance, for this class).
        # Different lime versions store `explanation.score` differently:
        #   - dict {label: score}  -> newer versions, per-class
        #   - single float          -> older versions, only for the last
        #                              explained label (usually top_labels[0])
        # Handle both safely.
        score_attr = getattr(explanation, "score", None)
        if isinstance(score_attr, dict):
            fidelity = float(score_attr.get(c, np.nan))
        elif isinstance(score_attr, (int, float)):
            # only meaningful for the class LIME actually fit last -- usually
            # the top predicted class. For other classes we don't have a
            # per-class score in this lime version, so mark as NaN.
            fidelity = float(score_attr) if c == pred_idx else np.nan
        else:
            fidelity = np.nan

        fidelity_str = f"{fidelity:.3f}" if not np.isnan(fidelity) else "N/A"
        axes[row, c + 1].imshow(vis)
        axes[row, c + 1].set_title(
            f"{class_names[c]}\n"
            f"Model conf: {preds[c]*100:.2f}%\n"
            f"LIME fidelity (R2): {fidelity_str}"
        )
        axes[row, c + 1].axis("off")

        lime_records.append({
            "image_index": idx,
            "true_label": class_names[true_label],
            "pred_label": class_names[pred_idx],
            "explained_class": class_names[c],
            "model_confidence": float(preds[c]),   # classifier's own softmax output
            "lime_fidelity_r2": fidelity,           # LIME's own local-model quality metric
        })

plt.tight_layout()
plt.savefig("/kaggle/working/lime_all_classes.png", dpi=600)
plt.show()

pd.DataFrame(lime_records).to_csv("/kaggle/working/lime_confidence.csv", index=False)
print("Saved: lime_all_classes.png, lime_confidence.csv")
print("\nNote: 'model_confidence' = classifier softmax output (NOT LIME's output).")
print("      'lime_fidelity_r2'  = LIME's own local surrogate model quality (R^2).")
print("      NaN mane ei lime version-e oi class-er jonno alada score store hoyni")
print("      (only the predicted class's fidelity is available in this version).")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import shap

def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)

SAMPLES_PER_CLASS = 1  # dorkar hole barao
val_labels_arr = np.array(val_labels)
selected_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    if len(class_indices) == 0:
        print(f"Warning: no validation samples for class '{class_names[class_idx]}'")
        continue
    selected_indices.extend(class_indices[:SAMPLES_PER_CLASS].tolist())

sample_imgs = []
sample_true_labels = []
for idx in selected_indices:
    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    sample_imgs.append(img.numpy())
    sample_true_labels.append(true_label)
sample_imgs = np.array(sample_imgs)

masker = shap.maskers.Image("blur(32,32)", (IMG_SIZE, IMG_SIZE, 3))
shap_explainer = shap.Explainer(predict_fn, masker, output_names=class_names)

SHAP_MAX_EVALS = 300

shap_values = shap_explainer(
    sample_imgs,
    max_evals=SHAP_MAX_EVALS,
    batch_size=50,
    outputs=shap.Explanation.argsort.flip[:len(class_names)]
)
# shap_values.values shape: (n_images, H, W, C, n_classes)
# shap_values.output_names gives the class name per output slot, PER IMAGE
# (because outputs were sorted most-likely-first per image)

preds_all = predict_fn(sample_imgs)

n_rows = len(selected_indices)
n_cols = len(class_names) + 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

shap_records = []

for row, idx in enumerate(selected_indices):
    img_disp = sample_imgs[row] / 255.0
    axes[row, 0].imshow(img_disp)
    pred_idx = int(np.argmax(preds_all[row]))
    axes[row, 0].set_title(
        f"Original\nTrue: {class_names[sample_true_labels[row]]}\n"
        f"Pred: {class_names[pred_idx]} ({preds_all[row, pred_idx]*100:.2f}%)"
    )
    axes[row, 0].axis("off")

    # Per-row (per-image) heatmaps for ALL classes, channel-summed
    row_heatmaps = {}
    for slot in range(len(class_names)):
        cname = shap_values.output_names[row][slot] if hasattr(shap_values.output_names[row], "__getitem__") \
                else shap_values.output_names[slot]
        heat = shap_values.values[row][..., slot]      # (H, W, C)
        heat = heat.sum(axis=-1)                        # (H, W), signed
        row_heatmaps[cname] = heat

    # PER-IMAGE (per-row) color scale -- this is the key fix.
    row_max_abs = max(np.abs(h).max() for h in row_heatmaps.values()) + 1e-12

    for c, cname in enumerate(class_names):
        heat = row_heatmaps.get(cname, np.zeros(sample_imgs[row].shape[:2]))
        axes[row, c + 1].imshow(img_disp)
        im = axes[row, c + 1].imshow(
            heat, cmap="coolwarm", alpha=0.6,
            vmin=-row_max_abs, vmax=row_max_abs
        )
        mean_abs_shap = float(np.abs(heat).mean())
        model_conf = float(preds_all[row, c])

        axes[row, c + 1].set_title(
            f"{cname}\nModel conf: {model_conf*100:.2f}%\n"
            f"Mean |SHAP|: {mean_abs_shap:.5f}"
        )
        axes[row, c + 1].axis("off")

        shap_records.append({
            "image_index": idx,
            "true_label": class_names[sample_true_labels[row]],
            "pred_label": class_names[pred_idx],
            "explained_class": cname,
            "model_confidence": model_conf,     # classifier's own softmax output
            "mean_abs_shap": mean_abs_shap,      # SHAP's own attribution magnitude
        })

    # small shared colorbar per row
    cbar = fig.colorbar(im, ax=axes[row, 1:].tolist(), shrink=0.7, pad=0.01)
    cbar.set_label("SHAP value (row-normalized)")

plt.savefig("/kaggle/working/shap_all_classes.png", dpi=600, bbox_inches="tight")
plt.show()

pd.DataFrame(shap_records).to_csv("/kaggle/working/shap_confidence.csv", index=False)
print("Saved: shap_all_classes.png, shap_confidence.csv")
print("\nNote: 'model_confidence' = classifier softmax output (NOT SHAP's output).")
print("      'mean_abs_shap'     = SHAP's own attribution magnitude for that class,")
print("      normalized PER IMAGE so non-predicted classes stay visible.")

In [ ]:

import os
import cv2
import numpy as np
import pandas as pd
from lime import lime_image
import shap

# ========================== CONFIG ==========================
CHECKPOINT_PATH = "/kaggle/working/all_methods_causal_metrics_progress.csv"
SUMMARY_PATH    = "/kaggle/working/all_methods_causal_metrics_per_class_summary.csv"

N_PER_CLASS       = 10     # koyta image per class evaluate korbe. Barale
                            # result stronger/statistically valid hobe kintu
                            # LIME+SHAP shomoy onek barbe (slow perturbation methods).
                            # Paper er jonno minimum 10-15 recommend kori.
PATCH_SIZE        = 16
N_STEPS           = 20
LIME_NUM_SAMPLES  = 1000
SHAP_MAX_EVALS    = 300

# ========================== CAUSAL METRIC UTILITIES ==========================
def get_patch_order(heatmap_224, patch_size=16):
    h, w = heatmap_224.shape
    n_h, n_w = h // patch_size, w // patch_size
    patch_scores = np.zeros((n_h, n_w))
    for i in range(n_h):
        for j in range(n_w):
            patch_scores[i, j] = heatmap_224[
                i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size
            ].mean()
    flat_order = np.argsort(-patch_scores.ravel())
    order = np.stack(np.unravel_index(flat_order, patch_scores.shape), axis=1)
    return order


def causal_metric(img_224, heatmap_224, pred_index, mode="insertion",
                   patch_size=16, n_steps=20):
    order = get_patch_order(heatmap_224, patch_size)
    n_patches = len(order)
    step = max(1, n_patches // n_steps)

    blurred = cv2.GaussianBlur(img_224, (31, 31), 0)
    canvas = blurred.copy() if mode == "insertion" else img_224.copy()

    scores, fractions = [], []
    for idx in range(0, n_patches + step, step):
        idx = min(idx, n_patches)
        for (pi, pj) in order[:idx]:
            r0, r1 = pi*patch_size, (pi+1)*patch_size
            c0, c1 = pj*patch_size, (pj+1)*patch_size
            if mode == "insertion":
                canvas[r0:r1, c0:c1] = img_224[r0:r1, c0:c1]
            else:
                canvas[r0:r1, c0:c1] = blurred[r0:r1, c0:c1]

        pred = model.predict(canvas[np.newaxis, ...], verbose=0)[0]
        scores.append(pred[pred_index])
        fractions.append(idx / n_patches)
        if idx == n_patches:
            break

    return float(np.trapz(scores, fractions))


def lime_to_heatmap(explanation, label, img_shape):
    segments = explanation.segments
    heatmap = np.zeros(img_shape[:2], dtype=np.float32)
    for seg_id, weight in explanation.local_exp[label]:
        heatmap[segments == seg_id] = weight
    heatmap = np.maximum(heatmap, 0)   # positive (supporting) evidence only
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    return heatmap


def predict_fn(images):
    images = np.array(images, dtype=np.float32)
    return model.predict(images, verbose=0)


# ========================== BUILD BALANCED EVAL SET ==========================
val_labels_arr = np.array(val_labels)
eval_indices = []
for class_idx in range(len(class_names)):
    class_indices = np.where(val_labels_arr == class_idx)[0]
    n_take = min(N_PER_CLASS, len(class_indices))
    if n_take < N_PER_CLASS:
        print(f"Warning: class '{class_names[class_idx]}' has only {len(class_indices)} "
              f"val samples, using all of them.")
    eval_indices.extend(class_indices[:n_take].tolist())

print(f"Total evaluation images: {len(eval_indices)} "
      f"(~{N_PER_CLASS} per class x {len(class_names)} classes)")

# ========================== RESUME FROM CHECKPOINT ==========================
result_columns = [
    "image_index", "true_label", "pred_label", "confidence",
    "GradCAM_Insertion_AUC",   "GradCAM_Deletion_AUC",
    "GradCAM++_Insertion_AUC", "GradCAM++_Deletion_AUC",
    "LIME_Insertion_AUC",      "LIME_Deletion_AUC",
    "SHAP_Insertion_AUC",      "SHAP_Deletion_AUC",
]

if os.path.exists(CHECKPOINT_PATH):
    results_df = pd.read_csv(CHECKPOINT_PATH)
    done_indices = set(results_df["image_index"].tolist())
    print(f"Resuming... {len(done_indices)}/{len(eval_indices)} images already done")
else:
    results_df = pd.DataFrame(columns=result_columns)
    done_indices = set()

# ========================== EXPLAINERS (build once) ==========================
lime_explainer = lime_image.LimeImageExplainer()
masker = shap.maskers.Image("blur(32,32)", (IMG_SIZE, IMG_SIZE, 3))
shap_explainer = shap.Explainer(predict_fn, masker, output_names=class_names)

# ========================== MAIN LOOP (all 4 methods, same image, same pred_idx) ==========================
for i, idx in enumerate(eval_indices):
    if idx in done_indices:
        continue

    path, true_label = val_paths[idx], val_labels[idx]
    img, _ = load_image(path, true_label)
    img_np = img.numpy()
    img_batch = img_np[np.newaxis, ...]

    preds = predict_fn(img_batch)[0]
    pred_idx = int(np.argmax(preds))
    confidence = float(preds[pred_idx])

    # ---------------- Grad-CAM ----------------
    heatmap_gc, _, _ = compute_gradcam(img_batch, pred_index=pred_idx)
    heatmap_gc_224 = cv2.resize(heatmap_gc, (IMG_SIZE, IMG_SIZE))
    ins_gc = causal_metric(img_np, heatmap_gc_224, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_gc = causal_metric(img_np, heatmap_gc_224, pred_idx, "deletion",  PATCH_SIZE, N_STEPS)

    # ---------------- Grad-CAM++ ----------------
    heatmap_gcpp, _, _ = compute_gradcam_plusplus(img_batch, pred_index=pred_idx)
    heatmap_gcpp_224 = cv2.resize(heatmap_gcpp, (IMG_SIZE, IMG_SIZE))
    ins_gcpp = causal_metric(img_np, heatmap_gcpp_224, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_gcpp = causal_metric(img_np, heatmap_gcpp_224, pred_idx, "deletion",  PATCH_SIZE, N_STEPS)

    # ---------------- LIME ----------------
    explanation = lime_explainer.explain_instance(
        img_np.astype(np.double), predict_fn,
        top_labels=len(class_names), hide_color=0, num_samples=LIME_NUM_SAMPLES
    )
    heatmap_lime = lime_to_heatmap(explanation, pred_idx, img_np.shape)
    ins_lime = causal_metric(img_np, heatmap_lime, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_lime = causal_metric(img_np, heatmap_lime, pred_idx, "deletion",  PATCH_SIZE, N_STEPS)

    # ---------------- SHAP ----------------
    sv = shap_explainer(
        img_batch, max_evals=SHAP_MAX_EVALS, batch_size=50,
        outputs=shap.Explanation.argsort.flip[:len(class_names)]
    )
    heatmap_shap = sv.values[0][..., 0]           # top predicted class map
    heatmap_shap = np.abs(heatmap_shap).sum(axis=-1)
    heatmap_shap = heatmap_shap / (heatmap_shap.max() + 1e-8)
    ins_shap = causal_metric(img_np, heatmap_shap, pred_idx, "insertion", PATCH_SIZE, N_STEPS)
    del_shap = causal_metric(img_np, heatmap_shap, pred_idx, "deletion",  PATCH_SIZE, N_STEPS)

    # ---------------- Save row ----------------
    results_df.loc[len(results_df)] = [
        idx, class_names[true_label], class_names[pred_idx], confidence,
        ins_gc, del_gc, ins_gcpp, del_gcpp,
        ins_lime, del_lime, ins_shap, del_shap
    ]
    results_df.to_csv(CHECKPOINT_PATH, index=False)

    print(f"[{i+1}/{len(eval_indices)}] idx={idx} true={class_names[true_label]} "
          f"pred={class_names[pred_idx]} conf={confidence*100:.2f}%  "
          f"GC(ins/del)={ins_gc:.3f}/{del_gc:.3f}  "
          f"GC++(ins/del)={ins_gcpp:.3f}/{del_gcpp:.3f}  "
          f"LIME(ins/del)={ins_lime:.3f}/{del_lime:.3f}  "
          f"SHAP(ins/del)={ins_shap:.3f}/{del_shap:.3f}")

# ========================== PER-CLASS SUMMARY TABLE ==========================
metric_cols = [c for c in results_df.columns if "AUC" in c]
per_class_summary = results_df.groupby("true_label")[metric_cols].mean().round(4)
per_class_summary.loc["Overall_Mean"] = results_df[metric_cols].mean().round(4)
per_class_summary.to_csv(SUMMARY_PATH)

print("\n================================================================")
print("PER-CLASS Insertion (↑ better) / Deletion (↓ better) AUC — all methods")
print("================================================================")
print(per_class_summary)
print(f"\nSaved: {CHECKPOINT_PATH}")
print(f"Saved: {SUMMARY_PATH}")